# Week 3 — Expand to 200+ images, build the training Dataset

Doubles the Week 1 dataset with more synthetic variety (different fonts and
render styles are cheap; the handout is right that variety is what makes a
model robust) and wraps everything in `src/dataset.py`'s `UrduOCRDataset`,
which — vs. the handout's version — normalises Urdu text, preprocesses
images, and applies train-time augmentation (see that file's docstring).


In [ ]:
# Run once per Colab session.
# If you opened this notebook from GitHub (File > Save a copy in GitHub, per Week 1),
# clone your repo so `src/` and the bundled fonts/corpus are available:

import os
if not os.path.exists('src'):
    # Replace with your own fork's URL if you're running this outside the
    # original solution repo.
    !git clone https://github.com/Rameesha-Malik/ML-SI-Solution-Urdu-OCR.git repo
    %cd repo

!pip install -q -r requirements.txt
import sys
sys.path.insert(0, 'src')
print('Setup complete.')


## Step 1 — Expand to 200+ images

In [ ]:
from synthetic import generate_dataset, merge_labels_csv

# Generate more synthetic variety on top of Week 1's set (different random seed).
stats = generate_dataset(
    output_dir='data/raw/synthetic',
    labels_csv='data/labels_synthetic_2.csv',
    num_images=150,
    pseudo_fraction=0.3,
    seed=123,
    image_prefix='synth_w3',
)
print(f'Added {stats.total} more images')

# Re-merge everything (synthetic batch 1 + batch 2 + your real photos) into labels.csv.
n = merge_labels_csv(
    ['data/labels_synthetic.csv', 'data/labels_synthetic_2.csv', 'data/labels_real.csv'],
    'data/labels.csv',
)
print(f'data/labels.csv now has {n} total labelled images')
print('Keep adding real photographed images to data/raw/ + data/labels_real.csv — '
      'real data matters more than more synthetic data past this point. See '
      'docs/ACCURACY_NOTES.md.')


Then preprocess anything newly added (skips files already in `data/processed/`):

In [ ]:
from preprocessing import preprocess_folder
count = preprocess_folder('data/raw', 'data/processed')
print(f'{count} images in data/processed/')


## Step 2 — Build the train/test split + Dataset class

`split_dataset` writes explicit `data/train.csv` / `data/test.csv` files (reproducible, diffable) rather than an in-memory `random_split` — Week 4's training script reads these directly.

In [ ]:
from dataset import split_dataset, UrduOCRDataset
from transformers import TrOCRProcessor

n_train, n_test = split_dataset(
    'data/labels.csv', 'data/train.csv', 'data/test.csv', test_size=0.15, seed=42,
)
print(f'Training samples: {n_train}')
print(f'Testing samples:  {n_test}')

processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')
train_dataset = UrduOCRDataset('data/train.csv', processor, train=True)
test_dataset = UrduOCRDataset('data/test.csv', processor, train=False)

sample = train_dataset[0]
print('Sample pixel_values shape:', sample['pixel_values'].shape)
print('Sample labels shape:', sample['labels'].shape)
print('Dataset is working correctly!')
